# Steps 3 & 4: Train CodeT5-Small to detect vulnerable functions

Runs on **Google Colab with a free GPU**: *Runtime → Change runtime type → T4 GPU*.

We fine-tune **CodeT5-Small** (the paper's best small model) on our cleaned
DiverseVul data and test it on two splits:
- `random`: test functions come from the same projects as training (paper: ~45% F1)
- `unseen`: test functions come from 80 projects the model never saw (paper: ~9% F1)

**Step 4:** set `RENAME_IDS = True` to replace project-specific names with `VAR_1`, `FUNC_1`, ...
before training and testing. Does that help on unseen projects?

We need 4 runs in total (change the settings cell, then run cells 4–8):

| Run | SPLIT | RENAME_IDS |
|---|---|---|
| 1 | random | False |
| 2 | unseen | False |
| 3 | random | True |
| 4 | unseen | True |

## 1. Check the GPU

In [ ]:
!nvidia-smi

## 2. Get the code and install libraries
If the GitHub repo is **private**, Colab can't clone it. Easiest fix: make the repo public
(GitHub → Settings → Danger Zone → Change visibility).

In [ ]:
REPO_URL = "https://github.com/aashitagrawal/diversevul-vuln-detection.git"
!git clone -q {REPO_URL} project
%cd project
!pip install -q -r requirements.txt transformers datasets accelerate
import sys
sys.path.append("src")

## 3. Download and clean the dataset (same scripts we ran on the laptop, ~5 min)

In [ ]:
!python src/download_data.py
!python src/clean_dedup.py

## 4. Settings

In [ ]:
SPLIT = "random"      # "random" or "unseen"
RENAME_IDS = False    # Step 4: True = rename identifiers to VAR_1, FUNC_1, ...
MODEL_NAME = "Salesforce/codet5-small"
MAX_LEN = 512         # max tokens per function (longer functions are cut)
EPOCHS = 3
BATCH_SIZE = 16
LEARNING_RATE = 5e-5
FP16 = True           # faster on T4. If the loss shows "nan", set this to False

## 5. Load the data and split it

In [ ]:
from load_data import load_diversevul
from splits import random_split, project_split, downsample_safe
from rename_ids import rename_identifiers

df = load_diversevul("data/diversevul_clean.json")
if RENAME_IDS:
    print("Renaming identifiers (~2 min)...")
    df["func"] = df["func"].apply(rename_identifiers)
if SPLIT == "random":
    train_df, val_df, test_df = random_split(df)
else:
    train_df, val_df, test_df = project_split(df)

# Keep all vulnerable functions + 3 safe ones per vulnerable one (faster training).
# Validation and test keep the real class balance.
train_df = downsample_safe(train_df, ratio=3)

for name, d in [("train", train_df), ("val", val_df), ("test", test_df)]:
    print(f"{name:5s} {len(d):6d} functions, {d['target'].sum():5d} vulnerable")

## 6. Tokenize (turn code text into token ids)

In [ ]:
from datasets import Dataset
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def to_dataset(d):
    ds = Dataset.from_dict({"text": d["func"].tolist(), "label": d["target"].tolist()})
    return ds.map(lambda b: tokenizer(b["text"], truncation=True, max_length=MAX_LEN),
                  batched=True, remove_columns=["text"])

train_ds = to_dataset(train_df)
val_ds = to_dataset(val_df)
test_ds = to_dataset(test_df)

## 7. Train

In [ ]:
import numpy as np
from sklearn.metrics import f1_score
from transformers import (AutoModelForSequenceClassification, DataCollatorWithPadding,
                          Trainer, TrainingArguments)

model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)

# The model also returns big internal tensors; keep only the prediction scores
# (otherwise evaluation runs out of memory)
def keep_logits(outputs, labels):
    return outputs[0] if isinstance(outputs, tuple) else outputs

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    return {"f1": f1_score(labels, logits.argmax(-1))}

args = TrainingArguments(
    output_dir="models/codet5",
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=64,
    learning_rate=LEARNING_RATE,
    fp16=FP16,
    eval_strategy="epoch",          # check validation F1 after every epoch
    save_strategy="epoch",
    save_total_limit=1,
    load_best_model_at_end=True,    # keep the epoch with the best validation F1
    metric_for_best_model="f1",
    logging_steps=200,
    report_to="none",
)

trainer = Trainer(model=model, args=args, train_dataset=train_ds, eval_dataset=val_ds,
                  data_collator=DataCollatorWithPadding(tokenizer),
                  compute_metrics=compute_metrics,
                  preprocess_logits_for_metrics=keep_logits)
trainer.train()

## 8. Pick the decision threshold, then test
We trained on 1 vulnerable : 3 safe, but real data is about 1 : 17. So "probability > 0.5"
is not the best cut-off. We pick the threshold that gives the best F1 on the
**validation** set, then use it on the **test** set (never tune on test).

In [ ]:
import json
import torch
from sklearn.metrics import accuracy_score, confusion_matrix, precision_score, recall_score

def vuln_probability(ds):
    logits = trainer.predict(ds).predictions
    return torch.softmax(torch.tensor(logits), dim=-1)[:, 1].numpy()

val_prob = vuln_probability(val_ds)
thresholds = np.arange(0.05, 1.0, 0.05)
val_f1s = [f1_score(val_df["target"], val_prob > t) for t in thresholds]
best_t = float(thresholds[int(np.argmax(val_f1s))])
print(f"Best threshold on validation: {best_t:.2f} (val F1 = {max(val_f1s):.3f})")

test_prob = vuln_probability(test_ds)
y_true = test_df["target"].values
y_pred = (test_prob > best_t).astype(int)
tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()

results = {
    "split": SPLIT,
    "rename_ids": RENAME_IDS,
    "threshold": best_t,
    "accuracy": accuracy_score(y_true, y_pred) * 100,
    "precision": precision_score(y_true, y_pred) * 100,
    "recall": recall_score(y_true, y_pred) * 100,
    "f1": f1_score(y_true, y_pred) * 100,
    "fpr": fp / (fp + tn) * 100,   # false positive rate: safe functions flagged as vulnerable
}
for k, v in results.items():
    print(f"{k:10s} {v:.2f}" if isinstance(v, float) else f"{k:10s} {v}")

RUN_NAME = f"{SPLIT}_{'renamed' if RENAME_IDS else 'baseline'}"
with open(f"results/{RUN_NAME}.json", "w") as f:
    json.dump(results, f, indent=2)

## 9. Done
Copy the printed numbers back to Claude (or download `results/<split>_<baseline|renamed>.json`
from the Files panel on the left). Then change the settings cell for the next run
(see the table at the top) and run cells 4–8 again. No need to re-download the data.